In [1]:
import re
import torch
import warnings

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE}")
print(f"torch: {torch.__version__}")

C:\Users\dhanush\AppData\Local\Temp\ipykernel_11172\3323785870.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader



Running on: cuda
torch: 2.11.0+cu128


In [2]:
PDF_PATH = "C:/Users/dhanush/OneDrive/Documents/Learn_Before_You_Invest_SEBI_RBI_AMFI_Guide-1.pdf"

loader = PyPDFLoader(PDF_PATH)
raw_pages = loader.load()

print(f"Loaded {len(raw_pages)} pages from the guide.")

Loaded 99 pages from the guide.


In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""],
)

doc_chunks = splitter.split_documents(raw_pages)

print(f"Split into {len(doc_chunks)} chunks.")

Split into 292 chunks.


In [4]:
embedder = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)

print("Embedding model ready.")

Embedding model ready.


In [5]:
vector_store = FAISS.from_documents(doc_chunks, embedder)

print(f"FAISS index built with {vector_store.index.ntotal} vectors.")

FAISS index built with 292 vectors.


In [6]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

print("Retriever ready.")

Retriever ready.


In [7]:
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=quant_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
)

# Must happen BEFORE pipeline() is built — pipeline() snapshots
# generation_config at construction time.
llm_model.generation_config.max_length = None
llm_model.generation_config.max_new_tokens = None

llm = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    return_full_text=False,
)

print(f"{LLM_NAME} loaded and ready.")

W0824 16:07:59.100000 11172 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Qwen/Qwen2.5-3B-Instruct loaded and ready.


In [8]:
ANSWER_SYSTEM = """You are a document-only assistant. You have ONE job: answer questions using ONLY the exact \
facts written in the Context block below. You are FORBIDDEN from using anything you already know.

ABSOLUTE RULES — no exceptions, ever:
1. If the answer is not explicitly stated in the Context, you MUST reply with exactly this sentence and \
nothing else: I don't have enough information in the provided documents.
2. Do not answer questions about people, celebrities, politicians, current events, sports, entertainment, \
general trivia, coding, or ANY topic — even if you know the answer — unless that exact fact is written in \
the Context.
3. Do not use outside knowledge to "fill gaps" in the Context. If the Context is incomplete, say you don't \
have enough information rather than completing it from memory.
4. Do not soften, hedge, or add disclaimers like "as an AI" or "based on general knowledge." Either the \
Context supports the answer, or you refuse.
5. Never explain WHY you don't have the information, never mention "the document's scope," never say \
what the context lacks. If you don't know, output ONLY the refusal sentence — nothing before or after it.
6. Keep answers under 120 words. One paragraph. No headings, no bullet points unless the Context itself is \
a list.
7. If the question asks about "my," "our," or "I" — anything personal to the user (their account, \
their portfolio, their location, their money) — you have no way to know this from a general reference \
document. Treat it the same as an off-topic question and refuse.

EXAMPLES OF CORRECT BEHAVIOR:

Context: "SIP allows investors to invest fixed amounts at regular intervals in mutual funds."
Question: What is SIP?
Correct answer: SIP lets investors put in a fixed amount at regular intervals into mutual funds.

Context: "SIP allows investors to invest fixed amounts at regular intervals in mutual funds."
Question: Who is the Prime Minister of India?
Correct answer: I don't have enough information in the provided documents.

Context: "SIP allows investors to invest fixed amounts at regular intervals in mutual funds."
Question: What is the capital of France?
Correct answer: I don't have enough information in the provided documents.

Context: "SIP allows investors to invest fixed amounts at regular intervals in mutual funds."
Question: What is the asset value of a famous investor?
Correct answer: I don't have enough information in the provided documents.

Context: "SIP allows investors to invest fixed amounts at regular intervals in mutual funds."
Question: Where is my asset location?
Correct answer: I don't have enough information in the provided documents.

Now follow these exact same rules for the real Context and Question below.

Context:
{context}

Question: {question}

Answer:"""

In [9]:
def ask_llm(messages: list, max_tokens: int = 150) -> str:
    """messages = [{"role": "system", "content": ...}, {"role": "user", "content": ...}]"""

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    raw = llm(
        prompt,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=0.1,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
    )[0]["generated_text"].strip()

    for marker in ("Final Answer:", "Answer:"):
        if marker in raw:
            raw = raw.split(marker)[-1].strip()

    junk_markers = (
        "Definition:", "Key Points", "Summary", "Conclusion",
        "Explanation", "Additional Information", "Note:",
        "This change", "This answer", "This response", "This rewrite",
    )
    for marker in junk_markers:
        if marker in raw:
            raw = raw.split(marker)[0].strip()

    seen, unique_sentences = set(), []
    for sentence in re.split(r"(?<=[.!?])\s+", raw):
        sentence = sentence.strip()
        if sentence and sentence not in seen:
            seen.add(sentence)
            unique_sentences.append(sentence)

    return " ".join(unique_sentences)

In [10]:
REJECT_DISTANCE = 0.85  # FAISS L2 distance on normalized vectors; lower = stricter

def generate_answer(question: str) -> dict:
    hits = vector_store.similarity_search_with_score(question, k=5)

    print("Retrieval distances:", [round(score, 3) for _, score in hits])

    if not hits or hits[0][1] > REJECT_DISTANCE:
        return {
            "question": question,
            "context": "",
            "answer": "I don't have enough information in the provided documents.",
        }

    context = "\n\n".join(doc.page_content for doc, _ in hits)
    prompt_text = ANSWER_SYSTEM.format(context=context, question=question)
    messages = [{"role": "user", "content": prompt_text}]
    answer = ask_llm(messages, max_tokens=150)

    return {"question": question, "context": context, "answer": answer}

In [11]:
CRITIQUE_SYSTEM = """You are grading an AI-generated answer against the context it was given. \
Score each dimension 1 (bad) to 5 (excellent), based only on the context — not outside knowledge. \
Do not explain, justify, or add commentary. Output exactly four lines:

Faithfulness: <score>
Completeness: <score>
Accuracy: <score>
Clarity: <score>"""

In [12]:
def critique_answer(question: str, context: str, answer: str) -> dict:
    messages = [
        {"role": "system", "content": CRITIQUE_SYSTEM},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer: {answer}"},
    ]
    verdict_text = ask_llm(messages, max_tokens=100)

    scores = re.findall(r":\s*([1-5])", verdict_text)
    labels = ["faithfulness", "completeness", "accuracy", "clarity"]

    verdict = {label: int(score) for label, score in zip(labels, scores)}
    for label in labels:
        verdict.setdefault(label, 0)

    verdict["overall_score"] = round(sum(verdict[l] for l in labels) / 4, 2)
    verdict["decision"] = "PASS" if verdict["overall_score"] >= 4 else "FAIL"

    return verdict

In [13]:
REWRITE_SYSTEM = """The answer you're given scored poorly on some dimensions. Rewrite it using ONLY \
the context provided — fix the weak points, keep everything already correct, remove repetition.

Output ONLY the improved answer itself. Never describe what you changed, why you changed it, or \
comment on the rewrite process — no phrases like "this answer," "this change," or "this response."

50-120 words, one paragraph, no headings."""

In [14]:
def rewrite_answer(question: str, context: str, answer: str, evaluation: dict) -> str:
    messages = [
        {"role": "system", "content": REWRITE_SYSTEM},
        {"role": "user", "content": (
            f"Context:\n{context}\n\nQuestion: {question}\n\n"
            f"Original answer: {answer}\n\nEvaluation: {evaluation}"
        )},
    ]
    improved = ask_llm(messages, max_tokens=180)

    return improved if len(improved) >= 20 else answer

In [15]:
def self_corrective_rag(question: str) -> dict:
    result = generate_answer(question)

    # If retrieval already rejected the question, stop here —
    # don't let critique/rewrite touch a refusal, or the model
    # may "helpfully" rewrite it into a hallucinated answer.
    if result["context"] == "":
        return {
            "question": result["question"],
            "context": "",
            "evaluation": None,
            "final_answer": result["answer"],
        }

    verdict = critique_answer(result["question"], result["context"], result["answer"])

    if verdict["decision"] == "PASS":
        final = result["answer"]
    else:
        final = rewrite_answer(result["question"], result["context"], result["answer"], verdict)

    return {
        "question": result["question"],
        "context": result["context"],
        "evaluation": verdict,
        "final_answer": final,
    }

In [ ]:
def chat():
    print("Finance Assistant — type \'exit\' to quit\n")
    while True:
        q = input("You: ")
        if q.lower() in ("exit", "quit", "bye"):
            print("Bot: THANKYOU FOR USING ME BUDDY!")
            break
        result = self_corrective_rag(q)
        print("Bot:", result["final_answer"])

chat()

Finance Assistant — type 'exit' to quit



KeyboardInterrupt: Interrupted by user